# 01 - Audit Dataset Cassava

Notebook ini hanya mengaudit data berlabel `train.csv` dan `train_images`. Tidak membuka atau memakai test set kompetisi. Jalankan dari atas ke bawah di Kaggle Notebook dengan input kompetisi Cassava terpasang. Hasil audit disimpan ke `/kaggle/working/audit_cassava` untuk diunduh.

Yang diperiksa: jumlah dan label, kecocokan CSV dengan berkas gambar, file rusak, dimensi citra, dan duplikat identik berdasarkan SHA-256. Audit tidak menghapus atau mengubah dataset sumber.

In [ ]:
from pathlib import Path
import hashlib
import json
import time

import pandas as pd
from PIL import Image, UnidentifiedImageError

DATA_ROOT = Path('/kaggle/input/competitions/cassava-leaf-disease-classification')
CSV_PATH = DATA_ROOT / 'train.csv'
IMAGE_DIR = DATA_ROOT / 'train_images'
LABEL_MAP_PATH = DATA_ROOT / 'label_num_to_disease_map.json'
OUT_DIR = Path('/kaggle/working/audit_cassava')
OUT_DIR.mkdir(parents=True, exist_ok=True)

for path in (CSV_PATH, IMAGE_DIR, LABEL_MAP_PATH):
    print(f'{path}: exists={path.exists()}')
    assert path.exists(), f'Input tidak ditemukan: {path}'

df = pd.read_csv(CSV_PATH)
with LABEL_MAP_PATH.open(encoding='utf-8') as f:
    label_map = json.load(f)

print('train.csv shape:', df.shape)
print('Kolom:', df.columns.tolist())
print('Peta label:', label_map)
display(df.head())

In [ ]:
required_columns = {'image_id', 'label'}
assert required_columns.issubset(df.columns), f'Kolom wajib tidak ada: {required_columns - set(df.columns)}'

assert not df['image_id'].isna().any(), 'Ada image_id kosong; periksa train.csv.'
df['image_id'] = df['image_id'].astype(str)
print('Baris CSV:', len(df))
print('image_id kosong:', int(df['image_id'].isna().sum()))
print('image_id duplikat:', int(df['image_id'].duplicated().sum()))
print('Label kosong:', int(df['label'].isna().sum()))
print('Label unik:', sorted(df['label'].dropna().unique().tolist()))

label_counts = (
    df['label'].value_counts(dropna=False).sort_index()
      .rename_axis('label').reset_index(name='count')
)
label_counts['disease'] = label_counts['label'].astype(str).map(label_map)
label_counts['fraction'] = label_counts['count'] / len(df)
display(label_counts)
label_counts.to_csv(OUT_DIR / 'class_distribution.csv', index=False)

In [ ]:
# Cocokkan nama file pada CSV dan folder gambar.
csv_names = set(df['image_id'])
disk_names = {p.name for p in IMAGE_DIR.iterdir() if p.is_file()}
missing_files = sorted(csv_names - disk_names)
unlisted_files = sorted(disk_names - csv_names)

print('Nama pada CSV tanpa file:', len(missing_files))
print('File gambar tanpa baris CSV:', len(unlisted_files))
print('File gambar di folder:', len(disk_names))

pd.DataFrame({'image_id': missing_files}).to_csv(OUT_DIR / 'missing_image_files.csv', index=False)
pd.DataFrame({'image_id': unlisted_files}).to_csv(OUT_DIR / 'unlisted_image_files.csv', index=False)

assert not df['image_id'].duplicated().any(), 'Ada image_id duplikat di CSV; periksa sebelum lanjut.'
assert not df['label'].isna().any(), 'Ada label kosong; periksa sebelum lanjut.'
valid_labels = {int(k) for k in label_map}
assert set(df['label'].astype(int).unique()) == valid_labels, 'Label pada CSV tidak sama dengan peta label.'
assert not missing_files, 'Ada file gambar yang dirujuk CSV tetapi tidak ditemukan.'

In [ ]:
# Verifikasi setiap citra, catat dimensi dan hash exact-file. Tidak ada berkas yang diubah.
rows = []
corrupt = []
started = time.time()

for i, row in enumerate(df.itertuples(index=False), start=1):
    image_id = str(getattr(row, 'image_id'))
    label = int(getattr(row, 'label'))
    path = IMAGE_DIR / image_id
    try:
        with Image.open(path) as im:
            im.verify()
        with Image.open(path) as im:
            width, height = im.size
            mode = im.mode
        digest = hashlib.sha256(path.read_bytes()).hexdigest()
        rows.append({
            'image_id': image_id, 'label': label, 'width': width,
            'height': height, 'mode': mode, 'sha256': digest
        })
    except (OSError, ValueError, UnidentifiedImageError) as exc:
        corrupt.append({'image_id': image_id, 'label': label, 'error': repr(exc)})
    if i % 1000 == 0 or i == len(df):
        print(f'{i:,}/{len(df):,} diperiksa')

audit_df = pd.DataFrame(rows)
corrupt_df = pd.DataFrame(corrupt, columns=['image_id', 'label', 'error'])
elapsed = time.time() - started
print(f'Valid: {len(audit_df):,}; rusak: {len(corrupt_df):,}; waktu: {elapsed/60:.1f} menit')
audit_df.to_csv(OUT_DIR / 'image_audit.csv', index=False)
corrupt_df.to_csv(OUT_DIR / 'corrupt_images.csv', index=False)

In [ ]:
# Kelompok hash identik menunjukkan file yang byte-nya sama persis.
duplicate_groups = (
    audit_df[audit_df.duplicated('sha256', keep=False)]
      .sort_values(['sha256', 'image_id'])
)
duplicate_groups.to_csv(OUT_DIR / 'exact_duplicate_groups.csv', index=False)
print('File dalam grup duplikat identik:', len(duplicate_groups))
print('Jumlah grup hash duplikat:', duplicate_groups['sha256'].nunique())
display(duplicate_groups.head(20))

dimension_summary = (
    audit_df.groupby(['width', 'height', 'mode']).size()
      .rename('count').reset_index().sort_values('count', ascending=False)
)
display(dimension_summary.head(20))
dimension_summary.to_csv(OUT_DIR / 'image_dimensions.csv', index=False)

In [ ]:
summary = {
    'data_root': str(DATA_ROOT),
    'csv_rows': int(len(df)),
    'unique_image_ids': int(df['image_id'].nunique()),
    'folder_image_files': int(len(disk_names)),
    'missing_files': int(len(missing_files)),
    'unlisted_files': int(len(unlisted_files)),
    'valid_images': int(len(audit_df)),
    'corrupt_images': int(len(corrupt_df)),
    'exact_duplicate_files_in_groups': int(len(duplicate_groups)),
    'exact_duplicate_hash_groups': int(duplicate_groups['sha256'].nunique()),
    'class_counts': {str(k): int(v) for k, v in df['label'].value_counts().sort_index().items()},
    'label_map': label_map,
    'elapsed_seconds': round(elapsed, 2),
}
(OUT_DIR / 'audit_summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
print(json.dumps(summary, indent=2))
print('Audit files saved to:', OUT_DIR)
print('Download audit_cassava from the Kaggle Notebook Output panel.')